# 🍗 Dynamic Incentive Optimizer v2 — DIO-KFC

**Applying Duolingo's Streak Psychology to KFC's Chicken Miles × Uplift Modeling (T-Learner)**

---

## このノートブックで行うこと

| Step | 内容 | モジュール |
|------|------|----------|
| 1 | 合成データ生成（10K顧客×4セグメント×段階的介入フロー） | `generate_data.py` |
| 2 | T-Learner学習（Model A + Model B）+ Qini曲線 + SHAP | `train_uplift.py` |
| 3 | CATEベース期待利益最大化 + 鉄板層自動カット | `optimize_incentive.py` |
| 4 | 増分ROI・無駄削減率・CATE分布 | `evaluate_roi.py` |
| 5 | 全図表生成（優先度A×2 + 優先度B×2） | `visualize.py` |

📝 **ビジネス背景・CRM戦略は [NOTE記事]() をご覧ください。**

---

## 0. 環境セットアップ

In [ ]:
# リポジトリのクローン
!git clone https://github.com/YOUR_USERNAME/dynamic-incentive-optimizer-v2.git
%cd dynamic-incentive-optimizer-v2

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from IPython.display import display, Image
import glob, os
print('✅ 環境セットアップ完了')

---
## 1. 合成データ生成

### 設計の核心: 段階的介入フロー

```
全顧客 (10,000人)
    ↓ サイクル崩れ検知
    ↓ 完全デジタル非金銭インセンティブ提示
        ├─ 反応あり (30%): is_churn=0 → コストゼロで防衛成功
        └─ 反応なし (70%): T-Learner対象
               ├─ Control群 (50%): discount=0.00
               └─ Treatment群 (50%): discount=0.01〜0.30
```

In [ ]:
from generate_data import generate_data, save_data, print_summary

df = generate_data()
save_data(df)
print_summary(df)

In [ ]:
display(df.head(10))

---
## 2. Stage 1: Uplift Modeling (T-Learner)

### なぜT-Learnerか？

$$\text{CATE}_i(d) = P_A(\text{churn}_i) - P_B(\text{churn}_i \mid d)$$

- **Model A** (Control): 自然離脱確率 $P_A$ を推定
- **Model B** (Treatment): 介入時離脱確率 $P_B(d)$ を推定  
- 差分のCATEが「割引の純粋な増分効果」を表し、**最適化の中核**に据えられる

In [ ]:
from train_uplift import run_training_pipeline

model_a, model_b, df_cate, metrics_train, qini = run_training_pipeline(df)

In [ ]:
# Uplift by Decile テーブル
print(f'\nQini係数: {qini["qini_coef"]:.4f}')
display(metrics_train['uplift_by_decile'])

In [ ]:
# キャリブレーション曲線
display(Image(filename='outputs/calibration_curves.png', width=650))

In [ ]:
# Qini曲線
display(Image(filename='outputs/qini_curve.png', width=700))

In [ ]:
# SHAP Summary: Model A vs Model B 対比
print('--- Model A (Control) ---')
display(Image(filename='outputs/shap_summary_model_a.png', width=700))
print('--- Model B (Treatment) ---')
display(Image(filename='outputs/shap_summary_model_b.png', width=700))

In [ ]:
# SHAP Waterfall: 3パターン
for path in sorted(glob.glob('outputs/shap_waterfall_*.png')):
    print(f'\n--- {os.path.basename(path)} ---')
    display(Image(filename=path, width=700))

---
## 3. Stage 2: CATEベース期待利益最大化

### 目的関数

$$\text{Profit}_i(d) = \underbrace{\text{CATE}_i(d) \times V_i \times (1-C)}_{\text{増分粗利}} - \underbrace{d \times V_i \times (1-P_B)}_{\text{割引コスト}} - \underbrace{\lambda \cdot V_i \cdot d^2 \cdot E_i}_{\text{ペナルティ}}$$

**T-Learner必然性のポイント:**  
CATEがゼロの鉄板層では第1項もゼロ → 割引コストが必ず利益を上回り → $d^* = 0$ が自動選択

In [ ]:
from optimize_incentive import run_optimization_pipeline

df_result = run_optimization_pipeline(model_a, model_b, df_cate)

In [ ]:
# 最適化結果の先頭確認
cols = ['user_id', 'current_rank', 'segment', 'avg_spend_recent',
        'p_natural_churn', 'cate_at_optimal', 'optimal_discount',
        'optimal_profit', 'uniform_profit', 'profit_improvement',
        'customer_type', 'is_iron_plate']
display(
    df_result[df_result['reacted_to_non_financial_incentive'] == 0][cols].head(15)
)

---
## 4. ビジネス評価 & 可視化

In [ ]:
from evaluate_roi import run_evaluation_pipeline

metrics = run_evaluation_pipeline(df_result, qini)

In [ ]:
# CATE分布（顧客タイプ別）
display(Image(filename='outputs/cate_distribution.png', width=750))

In [ ]:
# ROI比較
display(Image(filename='outputs/roi_comparison.png', width=600))

In [ ]:
# 無駄な割引カット効果
display(Image(filename='outputs/waste_reduction.png', width=850))

---
## 5. 追加可視化（優先度A: 利益カーブ & 優先度B: 感度分析）

In [ ]:
from optimize_incentive import sensitivity_analysis
from visualize import run_visualization_pipeline

# λ感度分析（時間がかかる場合はコメントアウト可）
sensitivity_df = sensitivity_analysis(model_a, model_b, df_cate)

# 全図表生成
run_visualization_pipeline(model_a, model_b, df_cate, df_result, sensitivity_df)

In [ ]:
# 割引率 vs 期待利益カーブ（ペナルティあり/なし）
display(Image(filename='outputs/profit_curve_with_penalty.png', width=900))

In [ ]:
# ペナルティ強度の感度分析
display(Image(filename='outputs/penalty_sensitivity.png', width=650))

---
## 📋 Summary

| 項目 | 内容 |
|------|------|
| 合成顧客数 | 10,000人 |
| セグメント | ファミリー / 単身ヘビー / 学生 / シニア |
| ランク体系 | レギュラー → ブロンズ → シルバー → ゴールド → プラチナ（90日判定） |
| 非金銭インセンティブ | 完全デジタル完結（バッジ・先行情報・デジタルスタンプ） |
| 予測モデル | T-Learner (LightGBM × 2) |
| 目的関数 | CATE × 粗利 − 割引コスト − ペナルティ（λV d² E） |
| 鉄板層判定 | $P_A < 0.10$ → $d^* = 0$ 自動強制 |

### Key Insight

**CATEがゼロの鉄板層では目的関数の第1項もゼロになり、割引コストが必ず利益を上回る。  
T-Learnerを使う必然性が最適化ロジック本体に直結している。**

---

📝 [NOTE記事]()  💻 [GitHub]()
